In [3]:
def is_variable(x):
    return x[0].islower() and "(" not in x

def resolve(term, subst):
    while is_variable(term) and term in subst:
        term = subst[term]
    if "(" in term:
        name, args = parse_term(term)
        resolved_args = [resolve(arg, subst) for arg in args]
        return f"{name}({', '.join(resolved_args)})"
    return term

def parse_term(expr):
    expr = expr.strip()
    idx = expr.index("(")
    name = expr[:idx].strip()
    inner = expr[idx + 1 : expr.rindex(")")]
    args = []
    depth = 0
    current = []
    for ch in inner:
        if ch == "(":
            depth += 1
            current.append(ch)
        elif ch == ")":
            depth -= 1
            current.append(ch)
        elif ch == "," and depth == 0:
            args.append("".join(current).strip())
            current = []
        else:
            current.append(ch)
    if current:
        args.append("".join(current).strip())
    return name, args

def occurs_check(var, term, subst):
    term = resolve(term, subst)
    if var == term:
        return True
    if "(" in term:
        _, args = parse_term(term)
        return any(occurs_check(var, arg, subst) for arg in args)
    return False

def unify(x, y, subst):
    x = resolve(x, subst)
    y = resolve(y, subst)

    if x == y:
        return True

    if is_variable(x):
        if occurs_check(x, y, subst):
            return False
        subst[x] = y
        return True

    if is_variable(y):
        if occurs_check(y, x, subst):
            return False
        subst[y] = x
        return True

    if "(" in x and "(" in y:
        name1, args1 = parse_term(x)
        name2, args2 = parse_term(y)
        if name1 != name2 or len(args1) != len(args2):
            return False
        for a, b in zip(args1, args2):
            if not unify(a, b, subst):
                return False
        return True

    return False

def main():
    p1 = input("Enter first predicate: ").strip()
    p2 = input("Enter second predicate: ").strip()

    if "(" not in p1 or "(" not in p2:
        print("FAILURE")
        return

    name1, a = parse_term(p1)
    name2, b = parse_term(p2)

    if name1 != name2 or len(a) != len(b):
        print("FAILURE")
        return

    subst = {}

    for i in range(len(a)):
        if not unify(a[i], b[i], subst):
            print("FAILURE")
            return

    if len(subst) == 0:
        print("NIL")
    else:
        final_subst = {k: resolve(v, subst) for k, v in subst.items()}
        print("Substitution:", final_subst)

main()


Enter first predicate:  P(x, f(y))
Enter second predicate:  P(A, f(B))


Substitution: {'x': 'A', 'y': 'B'}
